# Training a 100M Parameter Bengali Language Model

This notebook guides you through training a custom 100 million parameter language model from scratch on Google Colab's free T4 GPU.

**What you'll learn:**
- Setting up the environment
- Training a custom BPE tokenizer
- Preparing training data
- Pretraining the model
- Fine-tuning with instruction data
- Exporting for Android deployment

**Estimated time:** 4-8 hours (depending on dataset size)

## 1. Setup Environment

First, let's set up our environment and install dependencies.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set working directory
import os
WORK_DIR = '/content/drive/MyDrive/bengali_lm_100m'
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

In [ ]:
# Clone the repository (if not already present)
if not os.path.exists('src'):
    # If you have this in a git repo, clone it:
    # !git clone https://github.com/yourusername/bengali-lm-100m.git
    # %cd bengali-lm-100m
    
    # For now, we'll create the structure manually
    !mkdir -p configs data/raw data/processed data/examples src scripts
    print("✓ Directory structure created")
else:
    print("✓ Project already exists")

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers tokenizers datasets accelerate
!pip install -q pyyaml tqdm wandb
!pip install -q sentencepiece

# Optional: For export
!pip install -q onnx onnxruntime

print("✓ Packages installed")

In [ ]:
# Verify installation
import torch
import transformers

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Upload Training Data

Upload your Bengali text files to the `data/raw/` directory.

You can either:
- Upload files manually using the file browser (left sidebar)
- Download from a URL
- Use a dataset from Hugging Face

In [ ]:
# Example: Download Bengali text from a URL
# !wget -O data/raw/sample.txt "YOUR_DATA_URL_HERE"

# Example: Use Hugging Face datasets
# from datasets import load_dataset
# dataset = load_dataset("oscar", "unshuffled_deduplicated_bn", split="train[:10000]")
# 
# # Save to text file
# with open('data/raw/oscar_bn.txt', 'w', encoding='utf-8') as f:
#     for item in dataset:
#         f.write(item['text'] + '\n')

# For now, create a sample file for testing
sample_text = """এটি একটি নমুনা বাংলা টেক্সট ফাইল।
বাংলাদেশ দক্ষিণ এশিয়ার একটি দেশ।
শিক্ষা জাতির মেরুদণ্ড।
আমরা একটি ভাষা মডেল তৈরি করছি।
"""

with open('data/raw/sample.txt', 'w', encoding='utf-8') as f:
    f.write(sample_text * 100)  # Repeat for demo purposes

print("✓ Sample data created")
print("\nNote: For real training, add substantial Bengali text data (books, articles, etc.)")

In [ ]:
# Check uploaded data
!ls -lh data/raw/
!echo "\nTotal words:"
!wc -w data/raw/*.txt

## 3. Train Tokenizer

Train a BPE tokenizer on your Bengali text corpus.

In [ ]:
# Train tokenizer
!python scripts/train_tokenizer.py \
    --data_dir data/raw \
    --output_dir tokenizer \
    --vocab_size 32000 \
    --min_frequency 2

In [ ]:
# Test the tokenizer
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained('tokenizer')

# Test encoding/decoding
test_text = "এটি একটি পরীক্ষা বাক্য।"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)

print(f"Original: {test_text}")
print(f"Encoded: {encoded}")
print(f"Decoded: {decoded}")
print(f"\nVocab size: {len(tokenizer)}")

## 4. Prepare Training Data

Tokenize and chunk the text into training sequences.

In [ ]:
# Prepare data
!python scripts/prepare_data.py \
    --raw_data_dir data/raw \
    --output_dir data/processed \
    --tokenizer_path tokenizer \
    --block_size 2048 \
    --train_split 0.95

In [ ]:
# Check processed data
!ls -lh data/processed/*/
!cat data/processed/metadata.json

## 5. Pretrain the Model

Now let's train the 100M parameter language model!

In [ ]:
# Optional: Login to Weights & Biases for experiment tracking
# !pip install -q wandb
# import wandb
# wandb.login()

In [ ]:
# Start pretraining
# This will take several hours depending on your data size
!python scripts/pretrain.py \
    --config configs/model_config.yaml \
    --output_dir outputs/pretrain

### Monitor Training

Training will display progress with loss and learning rate. You can interrupt training at any time with Ctrl+C and resume later.

In [ ]:
# Check saved checkpoints
!ls -lh outputs/pretrain/

## 6. Test Text Generation

Let's test the pretrained model by generating some text.

In [ ]:
# Generate text with the pretrained model
!python scripts/generate.py \
    --model_path outputs/pretrain/best_model \
    --prompt "একবার এক" \
    --max_length 150 \
    --temperature 0.8 \
    --num_samples 3

## 7. Supervised Fine-Tuning (Optional)

If you have instruction-response pairs, you can fine-tune the model for better instruction-following.

In [ ]:
# Create sample SFT data
import json

sft_data = [
    {
        "instruction": "বাংলাদেশের রাজধানী কী?",
        "response": "বাংলাদেশের রাজধানী ঢাকা।"
    },
    {
        "instruction": "পদ্মা সেতু কোথায় অবস্থিত?",
        "response": "পদ্মা সেতু বাংলাদেশের মুন্সীগঞ্জ ও শরীয়তপুর জেলাকে সংযুক্ত করেছে।"
    },
    {
        "instruction": "বাংলা ভাষায় কত জন মানুষ কথা বলে?",
        "response": "প্রায় ২৩০ মিলিয়ন মানুষ বাংলা ভাষায় কথা বলে, যা এটিকে বিশ্বের সপ্তম সর্বাধিক কথ্য ভাষা করে তোলে।"
    },
]

# Save as JSONL
with open('data/examples/sft_data.jsonl', 'w', encoding='utf-8') as f:
    for item in sft_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print("✓ SFT data created")
print("\nNote: For real fine-tuning, create a larger dataset with diverse instructions")

In [ ]:
# Run supervised fine-tuning
!python scripts/sft.py \
    --config configs/model_config.yaml \
    --model_path outputs/pretrain/best_model \
    --data_path data/examples/sft_data.jsonl \
    --output_dir outputs/sft

In [ ]:
# Test the fine-tuned model
!python scripts/generate.py \
    --model_path outputs/sft/best_model \
    --prompt "নির্দেশ: বাংলাদেশের জাতীয় ফুল কী?\n\nউত্তর:" \
    --max_length 100

## 8. Export for Android

Export the model to formats compatible with Android deployment.

In [ ]:
# Export to ONNX and TFLite
!python scripts/export.py \
    --model_path outputs/pretrain/best_model \
    --output_dir exports \
    --format all \
    --quantize

In [ ]:
# Check exported files
!ls -lh exports/
!cat exports/model_info.json

## 9. Download Results

Download the trained model and exports for deployment.

In [ ]:
# Create a zip file with all important artifacts
!zip -r bengali_lm_100m_export.zip \
    exports/ \
    outputs/pretrain/best_model/ \
    tokenizer/ \
    configs/

print("\n✓ Export package created: bengali_lm_100m_export.zip")
print("Download it from the files panel (left sidebar)")

## 10. Model Information

Let's review the final model statistics.

In [ ]:
# Load and inspect the model
import sys
sys.path.append('.')

from src.model import CustomLMModel
from src.utils import print_model_info

model = CustomLMModel.from_pretrained('outputs/pretrain/best_model')
print_model_info(model)

## Summary

Congratulations! You've successfully:

1. ✓ Trained a custom BPE tokenizer for Bengali
2. ✓ Prepared training data
3. ✓ Pretrained a 100M parameter language model
4. ✓ (Optional) Fine-tuned with instruction data
5. ✓ Exported for Android deployment

### Next Steps:

1. **Improve the model:**
   - Add more training data
   - Train for more epochs
   - Tune hyperparameters

2. **Android Integration:**
   - Use ONNX Runtime Mobile or TensorFlow Lite
   - Implement inference in your Android app
   - Add UI for text generation

3. **Further Fine-tuning:**
   - Create domain-specific datasets
   - Fine-tune for specific tasks (QA, summarization, etc.)

### Resources:

- [ONNX Runtime Mobile](https://onnxruntime.ai/docs/tutorials/mobile/)
- [TensorFlow Lite](https://www.tensorflow.org/lite/guide/android)
- [Hugging Face Transformers](https://huggingface.co/docs/transformers/)

**Happy training! 🎉**